# Build 03-01 · Corrector inputs — targets ⨝ treatment, materialised per (version, split)

**Kernel: the analysis `.venv`** (`python3`). The one place the treatment join happens; the
corrector (03_02) and the re-evaluation (03_05) both read the artefact this notebook writes:

```
inputs/targets_<v>_<split>.parquet        (claim_id, date, observed)
   ⟕ (left join on claim_id) treatment    (decision [+ score])
   ──▶  mitigation/inputs/corrector_targets_<v>_<split>.parquet  (+ _meta.json)
```

Treatment source per version:
- **v3** — the v2 serving log's score file (`log_scores` kind, `logs/v2_score.parquet`):
  claim-level `score` + `decision` from the live model that generated v3's labels.
  Verified claim-unique on the company data (2026-09-02: `len == nunique`); a duplicate-event
  guard + collapse rule stays in §2 for reruns on refreshed exports.
- **v2** — the surviving **vehicle-status file** (§1 SOURCES). The real status vocabulary
  (user-described 2026-09-02): **fttl** (model fast-tracked → scrapped, label forced to 1),
  **repaired** / **total loss** (garage-verified outcomes), and three groups with **no usable
  outcome** — **awaiting authorisation** (repair decision still pending), **unrecovered**
  (stolen, never assessed: recorded total loss with neither garage verification nor an FTTL
  decision), and **NaN**. v2's original training swallowed all of them through the single
  observed column; **§4 drops the three unusable groups before the join**, so both splits and
  every downstream axis exclude them — the 03_02/03_03 **naive** axis thereby becomes the
  "v2 refit on known-outcome rows only" model. Verified to left-join onto the v2 training set
  with **zero unmatched rows** (2026-09). It carries `decision` only — the v1-era deciding
  **score is destroyed**, so v2's corrector_targets has no score column and rarity/pnu cannot
  run for v2 (thesis `tab:scheme-feasibility`); naive/transport can.

Built for **train + OOT** of each version, so 03_05 evaluates from the same artefact.

### 실행 전 설정

**커널**: analysis `.venv` (`python3`)

**바꿔야 할 것 (§1)**:
- `V2_STATUS_PATH` — 지금 `None`이라 **v2는 스킵되고 v3만 만들어집니다.** v2까지 만들려면 회사
  노트북에서 실제 vehicle-status 파일 경로를 채우세요 (Z: 드라이브 `.pkl`이면 joblib 덤프).
- `V2_STATUS_COLS` — 그 파일의 claim 컬럼명 / status 컬럼명 (`<<FILL IN>>` 채우기)
- `V2_STATUS_MAP` / `V2_STATUS_DROP` — 그 파일의 실제 status 문자열 (fttl/repaired/total loss
  등, `<<...>>` 값들). **실제 값을 그대로 옮겨 적기만 하고 추측/짐작으로 채우지 마세요** — §4가
  선언 안 된 값이 있으면 바로 에러를 냅니다.
- `MIN_COVERAGE`(기본 0.5)는 특별한 이유 없으면 안 건드려도 됩니다.

**선행 조건**: 없음 — mitigation 파이프라인에서 제일 먼저 도는 노트북입니다.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle

pd.set_option("display.width", 160)
print("ROOT =", ROOT)

figstyle.apply()
figstyle.FIG_DIR = ROOT / "figures" / "mitigation" / "03_01"
print("figures :", figstyle.FIG_DIR)

In [ ]:
# §1 — SOURCES / RUN_SPEC
ID_COL = "claim_id"
# Floor on the share of target rows the treatment source must cover. `matched.any()` is not a
# test: two UNRELATED integer key spaces still collide on a few rows and pass it — which is how
# v3's ID_CLAIM-keyed files joined "successfully" onto the claim-number-keyed v2 log
# (2026-09-03; since re-keyed by src/data/rekey_v3_claim_id.py). Lower it only with a reason
# printed next to it — e.g. a treatment source known to cover part of the window.
MIN_COVERAGE = 0.5
BUILD = {v: list(dict.fromkeys(["train", config.OOT_SPLIT[v]])) for v in ("v3", "v2")}

# v3 treatment: the v2 serving log's claim-level scores + decisions
V3_TREATMENT = config.path("log_scores", "v2")

# v2 treatment: the surviving vehicle-status file — company laptop only. Fill ALL of these in.
# NB if the file is a Z:-drive .pkl it is a JOBLIB dump (pd.read_pickle dies) — the reader below
# branches on the extension.
V2_STATUS_PATH = None                      # e.g. r"Z:/.../vehicle_status.pkl"; None -> skip v2
V2_STATUS_COLS = {"claim": "<<FILL IN>>",  # their claim-number column
                  "status": "<<FILL IN>>"} # their vehicle-status column

# The REAL status vocabulary (user-described 2026-09-02). Transcribe each value EXACTLY as it
# appears in the file — never guess a spelling; §4's audit fails loudly on any value declared
# in neither dict. Fill these placeholders in on the company laptop — real Allianz status
# strings are internal data and never belong in this repo's source.
#
# §4 separately normalises the literal STRING "nan" (confirmed 2026-09-13 — the column's real
# missing-value sentinel, NOT a true NaN) before this vocabulary is checked against it, so "nan"
# itself is never a value to declare here.
#
# KEPT — the only statuses with BOTH a treatment and a verified-or-forced outcome:
V2_STATUS_MAP = {
    # status value -> (decision, observed implied by the status)
    "<<fttl value>>":              (1, 1),  # model fast-tracked -> scrapped; label forced to 1
    "<<repaired value>>":          (0, 0),  # garage-verified repair (the negative class)
    "<<garage total loss value>>": (0, 1),  # garage-verified total loss
}
# DROPPED — no usable outcome, yet v2's original training swallowed them via the single
# observed column (status in {fttl, total loss, unrecovered} -> 1, else 0). NaN drops too (§4).
#   awaiting authorisation : repair-or-write-off decision still pending — outcome unknown
#                            (a DIFFERENT status from the "repaired" value above — do not
#                            conflate the two)
#   unrecovered            : stolen, never recovered — recorded total loss with neither a
#                            garage assessment nor an FTTL decision behind it
V2_STATUS_DROP = [
    "<<awaiting authorisation value>>",
    "<<unrecovered value>>",
]

print("build plan:", BUILD)
print("v3 treatment:", V3_TREATMENT)
print("v2 status   :", V2_STATUS_PATH or "(not set — v2 build will be skipped)")

In [ ]:
# §2 — helpers: the unique-claim gate, the collapse rule, the join+write step
def read_table(path) -> pd.DataFrame:
    """Parquet directly; a Z:-drive .pkl is a joblib dump, never pd.read_pickle."""
    p = str(path)
    if p.endswith(".parquet"):
        return pd.read_parquet(p)
    import joblib
    return joblib.load(p)


def norm_id(s: pd.Series) -> pd.Series:
    """claim_id as a stripped string, whatever dtype the source stored it in.

    The per-split exports and the log / vehicle-status sources do not agree on the claim_id
    dtype (int32 on one side, text on the other) and pandas refuses to merge across that.
    String is the safe common type: casting the other way would break a zero-padded or
    alphanumeric claim number. A float-typed read (any NaN in the column) renders as "12345.0",
    so it goes through Int64 first. This normalises the TYPE only — a padding or prefix
    difference still fails to match, which is what the coverage assertion in the join catches.
    """
    if pd.api.types.is_float_dtype(s):
        s = s.astype("Int64")
    return s.astype("string").str.strip()


def collapse_events(df: pd.DataFrame) -> pd.DataFrame:
    """One row per claim: a decision=1 event wins (its score); else the max-score event.

    Detection is the caller's unique-count gate; this runs ONLY when duplicates exist. Keeping
    a decision=0 event for a claim that was ever fast-tracked would misfile a scrapped car as
    garage-verified, and keeping the below-tau score of a scrapped claim would contradict the
    strict rule — sorting (decision desc, score desc) and keeping the first avoids both.
    """
    out = (df.sort_values(["decision", "score"], ascending=[False, False])
             .drop_duplicates(ID_COL, keep="first"))
    print(f"  collapsed {len(df):,} event rows -> {len(out):,} claims "
          f"({len(df) - len(out):,} duplicate events dropped)")
    return out


def build_corrector_targets(version: str, split: str, treatment: pd.DataFrame,
                            source_desc: str, section: str,
                            observed_check: pd.DataFrame | None = None,
                            extra_meta: dict | None = None,
                            min_coverage: float = MIN_COVERAGE) -> Path:
    """targets(split) ⟕ treatment -> corrector_targets parquet + meta. Returns the path.

    `section` names the CALLING cell (e.g. "sec3" for the v3 build, "sec4" for v2's) purely so
    every CSV this writes carries, in its filename, which section of this notebook produced it —
    it plays no role in the join itself.

    Rows the treatment frame does not cover are dropped by the left join's notna gate — for v2
    that now includes the status-dropped groups (§4 filters them out of `treatment` first), so
    `n_unmatched_dropped` counts status drops and genuine non-coverage together; `extra_meta`
    carries the per-status breakdown that tells them apart.
    """
    t = pd.read_parquet(config.split_path("targets", version, split))
    id_dtype = t[ID_COL].dtype          # the artefact keeps the targets' own dtype (below)
    t[ID_COL] = norm_id(t[ID_COL])
    treatment = treatment.assign(**{ID_COL: norm_id(treatment[ID_COL])})
    if observed_check is not None:
        observed_check = observed_check.assign(**{ID_COL: norm_id(observed_check[ID_COL])})

    m = t.merge(treatment, on=ID_COL, how="left", validate="one_to_one")
    matched = m["decision"].notna()
    n_un = int((~matched).sum())
    cov = float(matched.mean())
    print(f"{version} {split}: {len(m):,} target rows | coverage {cov:.1%} "
          f"({n_un:,} unmatched -> dropped)")
    # Always show both id samples: a key-space mismatch is visible to the eye long before any
    # statistic — and the floor below is what makes it fatal rather than a low coverage number.
    print(f"  {ID_COL} e.g.  targets {t[ID_COL].head(3).tolist()}  |  "
          f"treatment {treatment[ID_COL].head(3).tolist()}")

    # WHICH claim_ids the treatment source never covers — written before the coverage assert so
    # a failing floor still leaves this file behind for inspection. id_dtype restores each
    # claim_id to its own artefact's native form (the merge above ran on the normalised string).
    # Filename carries the notebook (0301) and section, per the file-naming convention this
    # project uses for every table a notebook writes.
    miss_path = None
    if n_un:
        # claim_id-bearing, one row per claim -- figures/mitigation/03_01/internal/, not
        # src/data/real/ (2026-09-16: data/ keeps only actual data artefacts now; every
        # informational table, id-bearing or not, lives under figures/ -- internal/ just
        # keeps this one visually apart from the rest of this notebook's output).
        miss_df = (m.loc[~matched, [ID_COL]]
                     .assign(**{ID_COL: lambda d: d[ID_COL].astype(id_dtype)}))
        miss_path = figstyle.save_table(
            miss_df, f"corrector_targets_{version}_{split}_0301_{section}_missing_claim_ids",
            index=False, subdir="internal")
        print(f"  {n_un:,} claim_id(s) missing from the treatment source -> {miss_path}")

    assert cov >= min_coverage, (
        f"{version} {split}: only {cov:.1%} of target rows matched the treatment on {ID_COL} "
        f"(floor {min_coverage:.0%}). The dtypes are aligned by norm_id, so the two sources key "
        f"differently — a row id vs the claim number (v3's ID_CLAIM, 2026-09-03), zero padding, "
        f"a prefix — compare the samples above and fix the SOURCE, never the join. If the "
        f"treatment source genuinely covers only part of this split, lower MIN_COVERAGE in §1 "
        f"with the reason written next to it.")

    # WHICH claims disagree, and on what — one row per mismatch, every column the two sources
    # brought (targets' own `observed`, treatment's `decision` [+ `score`], and whatever
    # `observed_check` carries: `observed_status` plus, for v2, the raw status string). This is
    # compared on `observed` vs `observed_status`, never `decision` — `decision` only says who
    # got matched, `observed`/`observed_status` are the two competing labels. THIS is the
    # dataset-sanity check: if a status that should be unambiguous (e.g. fttl -> observed_status
    # 1) disagrees with what was actually recorded (observed 0), the source data itself is
    # internally inconsistent — not something this notebook can fix, only surface.
    n_mismatch = None
    mismatch_path = None
    if observed_check is not None:
        chk = m.merge(observed_check, on=ID_COL, how="left")
        both = chk["decision"].notna() & chk["observed_status"].notna()
        disagree = both & (chk["observed"].astype("Int64") != chk["observed_status"].astype("Int64"))
        n_mismatch = int(disagree.sum())
        if n_mismatch:
            print(f"  !! observed (targets) vs status-implied outcome disagree on {n_mismatch} rows")
            # breakdown by whatever extra column(s) observed_check carries beyond the id and
            # observed_status itself — for v2 that is the raw status string, so this shows
            # EXACTLY which status value(s) are producing the disagreement (e.g. "fttl" rows
            # recorded as observed=0), not just a bare count.
            extra_cols = [c for c in observed_check.columns
                          if c not in (ID_COL, "observed_status")]
            if extra_cols:
                breakdown = (chk.loc[disagree, ["observed", "observed_status", *extra_cols]]
                                .value_counts().rename("n_claims").reset_index()
                                .sort_values("n_claims", ascending=False))
                print("  breakdown (targets' observed vs status-implied observed_status vs "
                      "raw source value):")
                print(breakdown.to_string(index=False))
            mismatch_path = figstyle.save_table(
                chk.loc[disagree].assign(**{ID_COL: lambda d: d[ID_COL].astype(id_dtype)}),
                f"corrector_targets_{version}_{split}_0301_{section}_observed_mismatches",
                index=False, subdir="internal")
            print(f"  -> {mismatch_path}")

    out = m.loc[matched].copy()
    out[ID_COL] = out[ID_COL].astype(id_dtype)   # normalisation was for the join only: every
    # downstream consumer joins this back onto features / score files, which still carry the
    # export dtype, so the written artefact must too (lossless — these rows came from targets).
    out["decision"] = out["decision"].astype(int)
    p = config.split_path("corrector_targets", version, split)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)
    p.with_name(p.stem + "_meta.json").write_text(json.dumps({
        "version": version, "split": split, "treatment_source": source_desc,
        "n_targets": int(len(m)), "n_unmatched_dropped": n_un,
        "n_out": int(len(out)), "has_score": bool("score" in out.columns),
        "n_scrapped": int(out["decision"].sum()),
        "n_observed_mismatch": n_mismatch,
        "missing_claim_ids_file": str(miss_path) if miss_path else None,
        "observed_mismatch_file": str(mismatch_path) if mismatch_path else None,
        "columns": list(out.columns), "id_dtype": str(id_dtype),
        **(extra_meta or {}),
    }, indent=2), encoding="utf-8")
    print(f"  -> {p.name}  (cols: {list(out.columns)})")
    return p

In [ ]:
# §3 — v3: treatment from the v2 serving log (score + decision)
tr3 = read_table(V3_TREATMENT)
for c in (ID_COL, "score", "decision"):
    assert c in tr3.columns, f"log_scores is missing {c!r} (canonical names — re-run 01_export_v2_logs)"
tr3 = tr3[[ID_COL, "score", "decision"]].copy()
tr3[ID_COL] = norm_id(tr3[ID_COL])       # dtype-align before the gate (see norm_id in §2)
n, u = len(tr3), tr3[ID_COL].nunique()
print(f"v2 log_scores: {n:,} rows / {u:,} unique {ID_COL}")
if n != u:
    tr3 = collapse_events(tr3)      # no-op gate as of 2026-09-02 (n == u on the company data)

built = []
for sp in BUILD["v3"]:
    built.append(build_corrector_targets("v3", sp, tr3, str(V3_TREATMENT), section="sec3"))

In [ ]:
# §3b — v3 missing claim_id: rate + whether it clusters (date only — targets has no channel
# column; src/schema.py declares no canonical FNOL/ENOL field, so a channel check would need the
# raw dataset under whatever real column name that version uses, not guessed here. Channel is an
# EXPLICITLY excluded confound for this thesis — decision 2026-09-13, see the top-level Notes).
for sp in BUILD["v3"]:
    p = config.split_path("corrector_targets", "v3", sp)
    meta_path = p.with_name(p.stem + "_meta.json")
    if not meta_path.is_file():
        print(f"v3 {sp}: not built yet — run §3 first")
        continue
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    miss_file = meta.get("missing_claim_ids_file")
    if not miss_file:
        print(f"v3 {sp}: 100% coverage — no missing claim_ids")
        continue

    t = pd.read_parquet(config.split_path("targets", "v3", sp))
    t[ID_COL] = norm_id(t[ID_COL])
    miss_ids = pd.read_csv(miss_file)
    miss_ids[ID_COL] = norm_id(miss_ids[ID_COL])
    miss = t.merge(miss_ids[[ID_COL]], on=ID_COL, how="inner")

    print(f"\nv3 {sp}: {len(miss):,} / {len(t):,} claim_id(s) missing from the v2 log "
          f"({len(miss) / len(t):.2%})")

    if "date" in t.columns:
        full_month = pd.to_datetime(t["date"]).dt.to_period("M")
        miss_month = pd.to_datetime(miss["date"]).dt.to_period("M")
        by_month = pd.DataFrame({
            "n_total": full_month.value_counts(),
            "n_missing": miss_month.value_counts(),
        }).fillna(0).astype(int).sort_index()
        by_month["missing_pct"] = by_month["n_missing"] / by_month["n_total"].replace(0, pd.NA)
        by_month["missing_pct"] = by_month["missing_pct"].map(
            lambda x: f"{x:.1%}" if pd.notna(x) else "—")
        print(f"  by month — a share concentrated in a few months points at a date-bounded log "
              f"(e.g. logging started/stopped), a roughly UNIFORM share across months points at "
              f"a key-space issue instead:")
        display(by_month)
        by_month_path = figstyle.save_table(
            by_month, f"corrector_targets_v3_{sp}_0301_sec3b_missing_by_month")
        print("  ->", by_month_path)
    else:
        print("  no `date` column in targets — cannot check a date pattern")

    if "observed" in miss.columns:
        print("  observed value among missing claims vs the full split (share skewed toward one "
              "label would suggest the log systematically excludes one outcome, not a random gap):")
        obs_mix = pd.DataFrame({
            "missing": miss["observed"].value_counts(normalize=True),
            "full_split": t["observed"].value_counts(normalize=True),
        }).fillna(0).round(4)
        display(obs_mix)
        obs_mix_path = figstyle.save_table(
            obs_mix, f"corrector_targets_v3_{sp}_0301_sec3b_missing_observed_mix")
        print("  ->", obs_mix_path)

In [ ]:
# §4 — v2: treatment from the vehicle-status file (decision only — no v1-era score)
if V2_STATUS_PATH is None:
    print("V2_STATUS_PATH not set — v2 build skipped (fill §1 SOURCES on the company laptop)")
else:
    st = read_table(V2_STATUS_PATH)
    claim_c, status_c = V2_STATUS_COLS["claim"], V2_STATUS_COLS["status"]
    for c in (claim_c, status_c):
        assert c in st.columns, f"status file has no column {c!r} — fix V2_STATUS_COLS"
    st = st[[claim_c, status_c]].rename(columns={claim_c: ID_COL}).copy()
    st[ID_COL] = norm_id(st[ID_COL])     # dtype-align before the gate (see norm_id in §2)

    # The status column's missing-value sentinel is the literal STRING "nan" (confirmed
    # 2026-09-13), not a real NaN — .isna()/.dropna() below do not catch it, so left alone it
    # would surface as an "undeclared status value" rather than as a drop. Normalise it to a
    # real NaN FIRST, once, so every .isna() check downstream (the audit, the drop mask, the
    # drop-count table) works the way it reads.
    str_nan_mask = st[status_c] == "nan"
    n_str_nan = int(str_nan_mask.sum())
    if n_str_nan:
        st.loc[str_nan_mask, status_c] = None
        print(f"normalised {n_str_nan:,} literal-string 'nan' status value(s) to real NaN")

    # -- status audit: every value must be declared KEPT or DROPPED, nothing passes silently --
    counts = st[status_c].value_counts(dropna=False)
    print("status counts (full file):\n" + counts.to_string(), "\n")
    undeclared = sorted(set(counts.index.dropna()) - set(V2_STATUS_MAP) - set(V2_STATUS_DROP))
    assert not undeclared, (
        f"undeclared status values {undeclared[:10]} — add each to V2_STATUS_MAP (known outcome) "
        f"or V2_STATUS_DROP (no usable outcome) in §1; nothing is inferred")

    # -- the drop: no verified outcome -> excluded from training AND evaluation, here, once ----
    drop_mask = st[status_c].isna() | st[status_c].isin(V2_STATUS_DROP)
    drop_counts = {"<NaN>": int(st[status_c].isna().sum()),
                   **{s: int((st[status_c] == s).sum()) for s in V2_STATUS_DROP}}
    st = st.loc[~drop_mask].copy()
    print(f"dropped {int(drop_mask.sum()):,} rows with no usable outcome: {drop_counts}")

    st["decision"] = st[status_c].map({k: v[0] for k, v in V2_STATUS_MAP.items()}).astype(int)
    st["observed_status"] = st[status_c].map({k: v[1] for k, v in V2_STATUS_MAP.items()}).astype(int)
    assert (st["observed_status"] == 0).any(), (
        "no verified-negative (repaired) rows remain after the drop — the retrain would have no "
        "negative class. Either the repaired status value in V2_STATUS_MAP is misspelled, or the "
        "file genuinely has no repaired status; STOP and re-read the vocabulary off the file")

    n, u = len(st), st[ID_COL].nunique()
    kept_counts = {str(k): int(v) for k, v in st[status_c].value_counts().items()}
    print(f"vehicle-status kept: {n:,} rows / {u:,} unique {ID_COL} | {kept_counts}")
    assert n == u, "kept status rows are not claim-unique — decide a collapse rule before building"

    for sp in BUILD["v2"]:
        built.append(build_corrector_targets(
            "v2", sp, st[[ID_COL, "decision"]], str(V2_STATUS_PATH), section="sec4",
            # the raw status string rides along here (not just observed_status) so that if
            # build_corrector_targets ever finds a mismatch, the saved CSV shows WHY — the
            # actual vehicle-status value behind that claim's implied outcome, not just the 0/1.
            observed_check=st[[ID_COL, "observed_status", status_c]],
            extra_meta={"status_dropped": drop_counts, "status_kept": kept_counts}))

In [ ]:
# §5 — what exists now
rows = []
for v, sps in BUILD.items():
    for sp in sps:
        p = config.split_path("corrector_targets", v, sp)
        if p.is_file():
            meta = json.loads(p.with_name(p.stem + "_meta.json").read_text(encoding="utf-8"))
            n_targets = meta["n_targets"]
            n_un = meta["n_unmatched_dropped"]
            n_mis = meta["n_observed_mismatch"]
            rows.append({
                "version": v, "split": sp,
                "n_targets": n_targets, "n_out": meta["n_out"], "n_scrapped": meta["n_scrapped"],
                "has_score": meta["has_score"],
                "unmatched_dropped": n_un,
                # share of ALL target rows this split started with — the denominator the
                # MIN_COVERAGE floor in §1 is checked against (coverage = 1 - this).
                "unmatched_pct": f"{n_un / n_targets:.2%}" if n_targets else "—",
                "observed_mismatch": n_mis,
                # v2 only (v3 never passes observed_check, so n_mis is always None there) —
                # share of the MATCHED population (n_out), since mismatch is only ever computed
                # among rows that already survived the join (see §2's build_corrector_targets).
                "mismatch_pct": (f"{n_mis / meta['n_out']:.2%}"
                                 if n_mis is not None and meta["n_out"] else "—"),
            })
        else:
            rows.append({"version": v, "split": sp, "n_out": "(not built)"})
summary = pd.DataFrame(rows).set_index(["version", "split"])
display(summary)
summary_path = figstyle.save_table(summary, "corrector_targets_0301_sec5_summary")
print("->", summary_path)

## Notes

- **File-naming convention (2026-09-14): every CSV this notebook writes names its own notebook
  and section**, e.g. `..._0301_sec3_missing_claim_ids.csv`, `..._0301_sec5_summary.csv` —
  `0301` = this notebook; `secN`/`secNb`/`secNc` matches this notebook's own `§N` headers.
  **2026-09-16: every CSV this notebook writes moved to `figures/mitigation/03_01/`, via
  `figstyle.save_table()`** — `src/data/real/` now holds only the real data artefacts
  (`corrector_targets` parquet + meta), never an informational table. The two per-row
  diagnostics (`missing_claim_ids`, `observed_mismatches` — real claim identifiers, one row
  per claim) go through `subdir="internal"`, landing in `figures/mitigation/03_01/internal/`,
  visually apart from the three AGGREGATE diagnostics (`missing_by_month`,
  `missing_observed_mix`, `sec5_summary` — no `claim_id` column, counts only) that save flat
  into `figures/mitigation/03_01/` alongside every other notebook's diagnostic tables.
- Unmatched rows (targets rows the treatment source never covers) are **dropped here, once** —
  every downstream consumer then works on the same treated population.
- **Which claim_ids are unmatched** is now written out too: whenever `n_unmatched_dropped > 0`,
  `build_corrector_targets` saves `corrector_targets_<v>_<split>_0301_<section>_missing_claim_ids.csv`
  next to the parquet (one `claim_id` column, in that split's own dtype) and records the path in
  `_meta.json` (`missing_claim_ids_file`) — `section` is `"sec3"` for v3's call, `"sec4"` for
  v2's. For v3 this is exactly "which v3 claim_ids are absent from the v2 serving log
  (`log_scores`)" — the file to open first when coverage is below 100%, before deciding whether
  it is a genuine gap or a key-space mismatch (see the `MIN_COVERAGE` comment in §1). **§3b**
  reads that file back and profiles it (share of the split, by-month count vs the full split,
  `observed` value mix — each also written to its own `..._0301_sec3b_...csv`) to help tell the
  two apart: concentrated in a few months or skewed toward one `observed` value points at a
  genuine, structured gap (logging window, a channel/segment the log never covered); a roughly
  uniform share across months and labels points at a key-space issue instead (like the
  2026-09-03 `ID_CLAIM` one).
- **Channel (FNOL/ENOL) is an explicitly excluded confound for the WHOLE thesis, not an
  oversight** (decision 2026-09-13): `targets` carries no such column and `src/schema.py`
  declares none canonical, so checking it would need the raw dataset under its real column
  name. Judged not a major driver and out of time budget to study properly — say so as a stated
  limitation wherever channel-mix could plausibly matter (e.g. the missing/mismatch diagnostics
  here, `README.md`'s reporting-lag section, `problem.md` §2.5 difficulty 6, and now
  `sec:concl-status` in the thesis), never silently drop the question.
- `§5`'s table reports `unmatched_pct` (share of `n_targets`, the same denominator `MIN_COVERAGE`
  checks) and `mismatch_pct` (share of `n_out`, since mismatch is only ever computed on rows that
  already survived the join — see below); saved whole to
  `figures/mitigation/03_01/corrector_targets_0301_sec5_summary.csv`.
- **Which claims disagree, and on what** (v2 only — `observed_check` is never passed for v3, so
  `n_observed_mismatch`/`observed_mismatch_file` are always `None` there): whenever
  `n_observed_mismatch > 0`, `build_corrector_targets` saves
  `corrector_targets_<v>_<split>_0301_sec4_observed_mismatches.csv` — one row per disagreeing
  claim, every column both sides brought (`claim_id`, targets' own `observed`, `decision`,
  `observed_status`, and the raw vehicle-status string §4 rides along in `observed_check`), plus
  a printed breakdown by that raw status value. This compares `observed` (the label v2 actually
  trained on) against `observed_status` (what the status file implies) — **not** `decision`,
  which only marks whether a row matched at all. It is computed strictly AFTER the
  unmatched/status-dropped rows are excluded (the `both` mask requires both sides non-null,
  which only matched rows satisfy) — so unmatched and mismatch are two different, non-overlapping
  populations: unmatched = "never joined at all", mismatch = "joined, but the two sources
  disagree on the label". A nonzero mismatch is a genuinely broken source row (e.g. a status
  meaning "fttl" recorded with `observed=0`) — not something this notebook can fix, only surface.
- **v2 status drop (2026-09-02)**: awaiting-authorisation, unrecovered, and NaN-status rows
  carry no verified outcome, so §4 removes them from the treatment frame **before**
  `build_corrector_targets` is even called. The drop therefore reaches **train and OOT alike**:
  the naive/transport retrains (03_02 → 03_03) fit on known-outcome rows only, and 03_05
  evaluates on the same filtered population. The per-status counts land in each `_meta.json`
  (`status_dropped` / `status_kept`). Those dropped rows are excluded from `treatment` before the
  join, so they fold into `n_unmatched_dropped` there (indistinguishable, inside that count, from
  genuine non-coverage — `status_dropped` is what separates them) and never reach the
  observed-mismatch comparison at all.
- v2's files carry **no `score` column**; 03_02 detects that and skips rarity/pnu with a printed
  reason. v3's carry score + decision, so all four schemes run.
- **No per-feature table lives in this notebook or `03_02`–`03_05`** (claim/axis/threshold-level
  data only), so none of them need the real-name/alias twin `00_SHAP.ipynb` does for individual
  Allianz feature names — that convention is specific to per-feature SHAP tables.
- Next: **03_02_reweight_mitigation.ipynb** (corrector), which now reads these files directly.